In [22]:
# Structures - construction industry
I_struc_industries = ['T41430']  # Construction

# Equipment - machinery, vehicles, electronics manufacturing
I_equip_industries = [
    'T26000',  # Manufacture of electronic components
    'T27000',  # Electrical equipment
    'T28000',  # Manufacture of machinery
    'T29000',  # Manufacture of motor vehicles and related parts
    'T30000',  # Manufacture of ships and other transport equipment
]

# IPP - software, R&D, publishing
I_ipp_industries = [
    'T58000',  # Publishing activities
    'T62630',  # IT and information service activities
    'T72001',  # Scientific research and development (market)
    'T72002',  # Scientific research and development (non-market)
]

# Organizational services - professional/business services (60% capitalized)
I_orga_industries = [
    'T69700',  # Legal/accounting/head offices/management consultancy
    'T71000',  # Architectural and engineering
    'T73000',  # Advertising and market research
    'T74750',  # Other professional/scientific/technical
    'T77000',  # Rental and leasing activities
    'T78000',  # Employment activities
    'T80820',  # Security/cleaning/other business services
]

supply = I_struc_industries + I_equip_industries + I_ipp_industries + I_orga_industries

In [23]:
# I_struc_industries_rev = ['41430 Construction-(Supply))']

# I_equip_industries_rev = [
#     '26000 Manufacture of electronic components-(Supply))',
#     '27000 Electrical equipment-(Supply))',
#     '28000 Manufacture of machinery-(Supply))',
#     '29000 Manufacture of motor vehicles and related parts-(Supply))',
#     '30000 Manufacture of ships and other transport equipment-(Supply))',
# ]

I_ipp_industries_rev = [
    '58000 Publishing activities-(Supply))',
    '62630 IT and information service activities-(Supply))',
    '72001 Scientific research and development (market)-(Supply))',
    '72002 Scientific research and development (non-market)-(Supply))',
]

I_orga_industries_rev = [
    '69700 Legal and accounting activities; activities of head offices; management consultanc-(Supply))',
    '71000 Architectural and engineering activities-(Supply))',
    '73000 Advertising and market research-(Supply))',
    '74750 Other professional, scientific and technical activities; veterinary activities-(Supply))',
    '77000 Rental and leasing activities-(Supply))',
    '78000 Employment activities-(Supply))',
    '80820 Security and investigation; services to buildings and landscape; other businness s-(Supply))',
]

supply_rev = I_orga_industries_rev + I_ipp_industries_rev

In [24]:
from py_files.setup import *
setup_notebook()

# %pip install git+https://github.com/alemartinello/dstapi
from IPython.display import display
from io import StringIO
from dstapi import DstApi
from functools import reduce

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# `Expenditure shares`

**Code:** 

1. **Part 1:** Get all industries in DST IO table and compute the individual share of investment versus consumption.

In [25]:
# 0. datasets
NAIO1F = DstApi('NAIO1F')
NAN1 = DstApi('NAN1')

# NAIO1F.tablesummary(language='en')
# NAIO1F.variable_levels('ANVENDELSE',language='en').text.unique()
# NAIO1F.variable_levels('TILGANG1',language='en')
# df = NAIO1F.variable_levels('ANVENDELSE',language='en')
# df.loc[df["id"].eq("AIAE"), "text"].iloc[0]

*`1. Get total output by industry`*

In [26]:
out_fetch = [
    'AA00000',  # Total use
    'ACPT',     # Household consumption total
    'ACO',      # Government consumption total
    'ABI',      # Gross fixed capital formation total
    'AE6000',   # Exports   
]

# 1. fetch
params_io = {
    'table' : 'NAIO1F',
    'format': 'BULK',
    'lang'  : 'en',
    'variables': [
        {'code': 'PRISENHED'    , 'values': ['V']},       
        {'code': 'Tid'          , 'values': ['*']},
        {'code': 'TILGANG1'     , 'values': ['P1_BP']},   
        {'code': 'TILGANG2'     , 'values': ['*']},     
        {'code': 'ANVENDELSE'   , 'values': out_fetch}, 
    ]
}

io_ = NAIO1F.get_data(params=params_io)
io_['INDHOLD'] = pd.to_numeric(io_['INDHOLD'], errors='coerce')

# 2. data manipulatui
io = io_[['TID','TILGANG2','ANVENDELSE','INDHOLD']].copy()

# 3. split tangibles and intangibles
io['intangible'] = io['TILGANG2'].isin(supply_rev)

In [27]:
io_T = io.pivot_table(index=['TID','TILGANG2','intangible'], columns='ANVENDELSE', values='INDHOLD').reset_index()

io_T = io_T.rename(columns={
    'Exports - (Use)'                           : 'X',
    'Gross fixed capiatal formation - (Use)'    : 'GFCF',
    'Household consumption expenditures (Use)'  : 'C',
    'Total Government Consumption-(Use)'        : 'G',
    'Total Use-(Use)'                           : 'TOTAL'
})

io_T['I_intan'] = io_T.intangible * io_T.GFCF
io_T['I_tan']   = (1-io_T.intangible) * io_T.GFCF

*add imports (different fetch)*

In [28]:
# 1. fetch
params_io = {
    'table' : 'NAIO1F',
    'format': 'BULK',
    'lang'  : 'en',
    'variables': [
        {'code': 'PRISENHED'    , 'values': ['V']},       
        {'code': 'Tid'          , 'values': ['*']},
        {'code': 'TILGANG1'     , 'values': ['P7AD2121']},   
        {'code': 'TILGANG2'     , 'values': ['*']},      
        {'code': 'ANVENDELSE'   , 'values': ['AIAE']}, 
    ]
}

io_imp = NAIO1F.get_data(params=params_io)
io_imp['INDHOLD'] = pd.to_numeric(io_imp['INDHOLD'], errors='coerce')
io_imp = io_imp.rename(columns={'INDHOLD' : 'M'})

# 2. merge with other
io_T = io_T.merge(io_imp[['TID','TILGANG2','M']], on=['TID','TILGANG2'], how='left')

*`2. compute breakdown by sector`*

In [33]:
# 1. final use shares
IO = io_T.copy()
IO['final_use'] = IO["C"] + IO["G"] + IO["GFCF"] + IO["X"] # - IO['M']

IO["C_share"]       = (IO["C"] + IO["G"]) / IO['final_use']     *100
IO["I_intan_share"] = IO["I_intan"] / IO['final_use']           *100
IO["I_tan_share"]   = IO["I_tan"] / IO['final_use']             *100
IO["X_share"]       = (IO["X"] - IO["M"]) / IO['final_use']     *100

*sanity*

In [38]:
# 1. print all for a specific year
year = 2022
cols = ['TILGANG2','C_share','I_intan_share','I_tan_share','X_share','TOTAL']

IO_year = IO[IO['TID'] == year]
IO_year = IO_year[cols].copy()
IO_year = IO_year.sort_values(by='I_tan_share', ascending=False)

IO_year['C_share'] = IO_year['C_share'].round(1)
IO_year['I_intan_share'] = IO_year['I_intan_share'].round(1)
IO_year['I_tan_share'] = IO_year['I_tan_share'].round(1)
IO_year['X_share'] = IO_year['X_share'].round(1)

IO_year.head(60) # messed up as there it double counting when using all TILGANG2

,TILGANG2,C_share,I_intan_share,I_tan_share,X_share,TOTAL
6578,41430 Construction-(Supply)),4.2,0.0,89.2,5.0,381762.627
6640,F Construction-(Supply)),4.2,0.0,89.2,5.0,381762.627
6595,68100 Buying and selling of real estate-(Supply)),15.5,0.0,84.2,-0.1,23364.008
6650,LA Real estate activities and renting of non-r...,21.8,0.0,77.6,0.3,105332.256
6654,MB Scientific research and development-(Supply)),4.5,0.0,46.7,6.6,51208.088
6574,33000 Repair and installation of machinery and...,1.3,0.0,40.2,20.9,24026.342
6646,"JA Publishing, television and radio broadcasti...",37.4,0.0,32.3,2.9,65172.453
6571,29000 Manufacture of motor vehicles and relate...,6.5,0.0,31.1,-787.1,6975.413
6648,JC IT and information service activities-(Supp...,4.5,0.0,29.2,45.6,127367.246
6589,59600 Motion picture and television programme ...,51.3,0.0,29.1,0.7,30783.220


*3. match with GDP*